In [2]:
import os
import numpy as np
import pandas as pd

import geopandas as gpd

## Export Calibration and Validation OBS datasets for each seed

In [3]:
COMPUTERNAME = os.environ['COMPUTERNAME']
print(f'Computer: {COMPUTERNAME}')

if COMPUTERNAME == 'BR_DELL':
    dir_font = os.path.join('/','run')
else:
    dir_font = os.path.join('/')

dir_base = os.path.join('/','media','arturo','T9','Data','Italy')

veneto_dir = os.path.join(dir_font,'media','arturo','T9','Data','shapes','Europa','Italy')

obs_base = os.path.join(dir_font,'media','arturo','T9','Data','Italy','Rain_Gauges_QC')

dir_cal = os.path.join('/','media','arturo','T9','Data','Italy','Rain_Gauges_QC','CAL_VAL', 'Calibration')
dir_val = os.path.join('/','media','arturo','T9','Data','Italy','Rain_Gauges_QC','CAL_VAL', 'Validation')

Computer: UNIPD_DELL


In [4]:
if os.path.exists(veneto_dir):
    ITALY = gpd.read_file(os.path.join(veneto_dir,'Italy_clear.geojson'))
else:
    raise SystemExit(f"File not found: {veneto_dir}")

In [5]:
list_remove = [
            'IT-820_1424_FTS_1440_QCv4.csv', 'IT-250_602781_FTS_1440_QCv4.csv', 
            'IT-250_602779_FTS_1440_QCv4.csv', 'IT-780_2370_FTS_1440_QCv4.csv', 
            'IT-750_450_FTS_1440_QCv4.csv', 'IT-520_TOS11000099_FTS_1440_QCv4.csv',
            'IT-520_TOS11000080_FTS_1440_QCv4.csv', 'IT-520_TOS11000072_FTS_1440_QCv4.csv',
            'IT-520_TOS11000060_FTS_1440_QCv4.csv', 'IT-520_TOS11000025_FTS_1440_QCv4.csv',
            'IT-520_TOS09001200_FTS_1440_QCv4.csv', 'IT-520_TOS02000237_FTS_1440_QCv4.csv',
            'IT-230_1200_FTS_1440_QCv4.csv'
            ]

METADATA = pd.read_csv(os.path.join(obs_base, 'data', 'METADATA', 'METADATA_FTS_QCv4_Case1_wAIRHO_v3_1dy.csv'))
METADATA["Lat"] = np.round(METADATA["Lat"], 6)
METADATA["Lon"] = np.round(METADATA["Lon"], 6)

METADATA = METADATA[~METADATA['File'].isin(list_remove)].reset_index(drop=True)

In [ ]:
# seeds_list = [7, 19, 31, 53, 89, 127, 211, 307, 401, 509, 613, 727, 839, 947, 1051]

primos_50 = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71,
            73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151,
            157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229]

np.random.seed(123)
aleatorios_50 = np.random.choice(range(1000, 10000), size=50, replace=False).tolist()

seeds_list = primos_50 + aleatorios_50

In [8]:
frac = 0.7

for seed in seeds_list:

    Q_cal_list = []
    Q_val_list = []

    for iso in METADATA['ISO'].unique():

        META_iso = METADATA[METADATA['ISO'] == iso]
        
        METADATA_70 = META_iso.sample(frac=frac, random_state=seed)
        METADATA_30 = META_iso.drop(METADATA_70.index)

        Q_cal_list.append(METADATA_70)
        Q_val_list.append(METADATA_30)

    METADATA_CAL = pd.concat(Q_cal_list, ignore_index=True)
    METADATA_VAL = pd.concat(Q_val_list, ignore_index=True)
    
    METADATA_CAL.to_csv(os.path.join(dir_cal, f'METADATA_CAL_seed{seed}.csv'), index=False)
    METADATA_VAL.to_csv(os.path.join(dir_val, f'METADATA_VAL_seed{seed}.csv'), index=False)